# 05. Predict — загрузить лучший чекпоинт → submission

Этот ноутбук не обучает модели.
Он принимает `best.pt` от любого из 4 основных ноутбуков и генерирует submission.csv.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [ ]:
# Локально: если репо уже установлен (uv sync / pip install -e .), эта ячейка — no-op.

# --- Colab ---
# !git clone --depth 1 https://github.com/DKazhekin/ITMO_Speech_Recognition_Course.git
# import sys
# sys.path.insert(0, "ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q num2words pyctcdecode jiwer tqdm librosa sentencepiece

# --- Kaggle ---
# import sys
# sys.path.insert(0, "/kaggle/working/ITMO_Speech_Recognition_Course/group-projects/gp1/src")
# !pip install -q num2words pyctcdecode jiwer tqdm librosa sentencepiece

print("Deps cell — fill in your platform block above if on Colab/Kaggle.")

## Пути (заполните вручную)

Задай абсолютные пути под свою среду. Все `FILL_ME_IN` должны быть реальными путями к train/dev/test и директории чекпоинтов.

In [ ]:
from pathlib import Path
import torch

# Fill in before running.
TRAIN_ROOT = Path("FILL_ME_IN")       # dir with train audio files (.wav / .mp3)
DEV_ROOT = Path("FILL_ME_IN")         # dir with dev audio files
TEST_ROOT: Path | None = None         # optional, set to Path("...") if have test data
TRAIN_CSV = Path("FILL_ME_IN")        # Kaggle-style CSV: filename, transcription, spk_id, gender, ext, samplerate
DEV_CSV = Path("FILL_ME_IN")
CKPT_ROOT = Path("FILL_ME_IN")        # where best.pt + meta.json will be saved

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")

Укажи путь к `best.pt` любой из четырёх основных моделей (смотри `meta.json` рядом с ним).

In [ ]:
from pathlib import Path
import json

CKPT = Path("FILL_ME_IN")  # e.g. CKPT_ROOT / "01_quartznet_.../trial_03/best.pt"
assert CKPT.exists(), f"Укажи путь к существующему best.pt: {CKPT}"

meta = json.loads((CKPT.parent / "meta.json").read_text())
print("Arch:", meta["arch"])
print("val_cer:", meta.get("val_cer", "N/A"))
print("hparams:", json.dumps(meta.get("hparams", {}), indent=2, default=str))


## Собрать модель по arch + hparams из meta

Диспатч по `meta["arch"]`: создаём нужный класс модели и загружаем веса.

In [ ]:
import torch
from gp1.train.checkpoint import load_checkpoint

arch = meta["arch"]
hp = meta.get("hparams", {})

if arch == "quartznet_10x4":
    from gp1.models.quartznet import QuartzNet10x4
    from gp1.text.vocab import CharVocab

    vocab = CharVocab()
    model = QuartzNet10x4(
        vocab_size=vocab.vocab_size,
        d_model=hp.get("d_model", 256),
        dropout=hp.get("dropout", 0.1),
        subsample_factor=hp.get("subsample_factor", 2),
    ).to(device)

elif arch == "crdnn":
    from gp1.models.crdnn import CRDNN
    from gp1.text.vocab import CharVocab

    vocab = CharVocab()
    model = CRDNN(
        vocab_size=vocab.vocab_size,
        d_cnn=hp.get("d_cnn", 64),
        rnn_hidden=hp.get("rnn_hidden", 256),
        rnn_layers=hp.get("rnn_layers", 2),
        dropout=hp.get("dropout", 0.15),
        subsample_factor=hp.get("subsample_factor", 1),
    ).to(device)

elif arch == "efficient_conformer":
    from gp1.models.efficient_conformer import EfficientConformer
    from gp1.text.vocab import CharVocab

    vocab = CharVocab()
    d_stages = hp.get("d_model_stages", [96, 128, 128])
    n_stages = hp.get("n_blocks_per_stage", [4, 4, 4])
    model = EfficientConformer(
        vocab_size=vocab.vocab_size,
        d_model_stages=tuple(d_stages),
        n_blocks_per_stage=tuple(n_stages),
        n_heads=hp.get("n_heads", 4),
        ff_ratio=hp.get("ff_ratio", 4),
        conv_kernel=hp.get("conv_kernel", 15),
        dropout=hp.get("dropout", 0.1),
    ).to(device)

elif arch == "fast_conformer_bpe":
    from gp1.models.fast_conformer_bpe import FastConformerBPE
    from gp1.text.vocab_bpe import BPEVocab

    bpe_model_path = CKPT_ROOT.parent / "bpe_models" / f"bpe_{hp.get('bpe_vocab_size', 256)}.model"
    vocab = BPEVocab(bpe_model_path)
    model = FastConformerBPE(
        vocab_size=vocab.vocab_size,
        d_model=hp.get("d_model", 96),
        n_blocks=hp.get("n_blocks", 16),
        n_heads=hp.get("n_heads", 4),
        ff_ratio=hp.get("ff_ratio", 4),
        conv_kernel=hp.get("conv_kernel", 9),
        dropout=hp.get("dropout", 0.1),
        subsample_factor=hp.get("subsample_factor", 4),
    ).to(device)

else:
    raise ValueError(f"Unknown arch: {arch}")

load_checkpoint(CKPT, model)
model.eval()
print(f"Loaded {arch} from {CKPT}")


## Инференс на test + submission

Запускаем greedy decode на всех тестовых файлах и пишем `submission.csv`.

In [ ]:
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.decoding.greedy import greedy_decode
from gp1.text.denormalize import words_to_digits
from gp1.submit.inference_utils import build_test_dataloader, write_submission

if TEST_ROOT is None:
    print("No test_root configured — set GP1_TEST_ROOT or place data in data/test/")
else:
    hop_length = hp.get("hop_length", 160)
    mel_extractor = LogMelFilterBanks(
        n_fft=hp.get("n_fft", 512),
        samplerate=hp.get("samplerate", 16000),
        hop_length=hop_length,
        win_length=hp.get("win_length", 400),
        n_mels=hp.get("n_mels", 80),
    ).to(device)

    test_records = records_from_csv(TEST_ROOT / "test.csv", TEST_ROOT)
    test_loader = build_test_dataloader(test_records, vocab=vocab)

    all_hyps = []
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // hop_length + 1).clamp(max=mel.size(-1)).long()
            )
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_hyps.extend(hyps)

    pairs = [
        (rec.audio_path.name, words_to_digits(h))
        for rec, h in zip(test_records, all_hyps)
    ]
    submission_path = CKPT_ROOT / "submission.csv"
    write_submission(pairs, submission_path)
    print(f"Submission saved ({len(pairs)} rows):", submission_path)
